In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

True

In [3]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [4]:

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [6]:
output = rag_pipeline("Do you have any earphones?")

In [7]:
output

{'data_object': RAGGenerationResponse(answer='Yes. We have kids wired earphones with a carry case (Volkano Ninja Kids Earphones, 3.5mm, red/blue) and two 3.5mm wired headphone options with microphone and volume control (a 2-pack for iPhone with MFi certification, and other wired 3.5mm compatible earphones). We also have wireless earbuds/headband options with charging cases.'),
 'answer': 'Yes. We have kids wired earphones with a carry case (Volkano Ninja Kids Earphones, 3.5mm, red/blue) and two 3.5mm wired headphone options with microphone and volume control (a 2-pack for iPhone with MFi certification, and other wired 3.5mm compatible earphones). We also have wireless earbuds/headband options with charging cases.',
 'question': 'Do you have any earphones?',
 'retrieved_context_ids': ['B0BBF2VC6X',
  'B0CH8DRD6K',
  'B09X9838WY',
  'B0BRV544MV',
  'B0CFHWF326'],
 'retrieved_context': ['Volkano Ninja Kids Earphones for Boys with Carry Case and Keyring - 3.5MM Stereo Jack - Wired Earbuds 

RAG pipeline with Grounding context

In [8]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the context")
    description: str = Field(description="Description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of references used to answer the question")

In [10]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [11]:
output = rag_pipeline("Do you have any earphones?")

In [12]:
output

{'data_object': RAGGenerationResponse(answer='Yes—there are earphones available:\n\n- Volkano Ninja Kids Earphones (wired, 3.5mm) with carry case and keyring (85dB safe volume)\n- 2 Pack iPhone Wired Headphones (3.5mm, Apple MFi certified) with microphone and volume control\n- Jesebang Bluetooth 5.2 Wireless Earbuds (with charging case, touch control, built-in mic, IP7)\n- Wekily Bluetooth 5.3 Wireless Earbuds (with charging case, 4-mic clear call, IPX7)\n- MUSICOZY Bluetooth 5.3 Headband Headphones with ENC mic (for sports/workouts)', references=[RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids Earphones for boys with carry case and keyring; wired 3.5mm stereo jack; safe volume limited to 85dB; includes mic and pause/play button.'), RAGUsedContext(id='B0CH8DRD6K', description='2 Pack iPhone Wired Headphones, Apple MFi certified; 3.5mm jack; built-in microphone and remote/volume control; compatible with 3.5mm devices.'), RAGUsedContext(id='B09X9838WY', description='Jeseb

In [13]:
print(output["answer"])

Yes—there are earphones available:

- Volkano Ninja Kids Earphones (wired, 3.5mm) with carry case and keyring (85dB safe volume)
- 2 Pack iPhone Wired Headphones (3.5mm, Apple MFi certified) with microphone and volume control
- Jesebang Bluetooth 5.2 Wireless Earbuds (with charging case, touch control, built-in mic, IP7)
- Wekily Bluetooth 5.3 Wireless Earbuds (with charging case, 4-mic clear call, IPX7)
- MUSICOZY Bluetooth 5.3 Headband Headphones with ENC mic (for sports/workouts)


In [14]:
output = rag_pipeline("Do you have any earphones?", top_k=10)

In [15]:

print(output["answer"])

Yes—there are several earphones/headphones available, including:
- Volkano Ninja Kids Earphones (wired, 3.5mm) with carry case
- 2 Pack iPhone Wired Headphones (3.5mm, with mic and volume control) 
- Jesebang Wireless Earbuds (Bluetooth 5.2, with charging case)
- Wekily Bluetooth 5.3 Wireless Earbuds (with charging case)
- MUSICOZY Bluetooth 5.3 Headband Headphones (ENC mic)
- pamu Wireless Earbuds (Bluetooth 5.2, ANC/ENC, charging case)
- Kids Wireless/ Wired Headphones (yellow, adjustable headband, Bluetooth 5.0 + 3.5mm)
- Bluet00th Sleep Headphones Headband (sleep mask headband)
- RUNAR Neckband Running Headphones (Bluetooth 5.0, sweat/rainproof)

And we also have a LONGFUN cleaning pen kit for AirPods Pro.
